In [7]:
from os import path
import polars as pl
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
from sklearn.svm import SVC

In [1]:
tabular_data_path = path.join("..", "..", "data", "tabular")
processed_csv = path.join(tabular_data_path, "processed.csv")

df = pl.read_csv(processed_csv)

target_col = "SONUÇ-2"

preop_features = [
    "YAŞ",
    "CİNSİYET",
    "TARAF",
    "LOKALİZASYON",
    "TOPLAM TAŞ YÜKÜ (CM2)",
    "SOLİTER BB",
    "RENAL ANOMALİ",
    "EK RENAL HASTALIK",
    "GEÇRİLMİŞ CERRAHİ",
    "TAS ANAMNEZI",
    "İKAB",
    "KREATİNİN",
    "ASA SKORU",
    "BT",
    "ÖZGEÇMİŞ",
    "GUY SCORE",
    "SATAVA S."
]

feature_cols = [c for c in preop_features if c in df.columns]

X = df.select(feature_cols).to_numpy()
y = df[target_col].cast(pl.Int64, strict=False).to_numpy() - 1

X = np.nan_to_num(X, nan=0.0)

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=492, stratify=y
)

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

n_neg = np.sum(y_train == 0)
n_pos = np.sum(y_train == 1)

scale_pos_weight = float(n_neg) / float(n_pos)

In [6]:
xgb_param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.3],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5],
    "reg_alpha": [0, 1, 5],
    "reg_lambda": [1, 5, 10],
    "gamma": [0, 1, 5],
}
xgb_base = xgb.XGBClassifier(
    random_state=492,
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight
)

xgb_grid = GridSearchCV(
    xgb_base,
    xgb_param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_scaled, y_train)
xgb_best = xgb_grid.best_estimator_
y_pred_xgb = xgb_best.predict(X_test_scaled)

Fitting 5 folds for each of 34992 candidates, totalling 174960 fits


In [4]:
svm_param_grid = {
    "C": [0.1, 1, 10, 100, 1000],
    "gamma": [0.001, 0.01, 0.1, 1, 10],
    "kernel": ["rbf", "poly", "sigmoid"],
    "class_weight": ["balanced"],
    "degree": [2, 3, 4]
}

svm_base = SVC(random_state=492)

svm_grid = GridSearchCV(
    svm_base,
    svm_param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=1
)

svm_grid.fit(X_train_scaled, y_train)
svm_best = svm_grid.best_estimator_
y_pred_svm = svm_best.predict(X_test_scaled)

Fitting 5 folds for each of 225 candidates, totalling 1125 fits


In [13]:
def evaluate_model(name, y_true, y_pred):
    print(f"--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"F1 Score: {f1_score(y_true, y_pred, average='weighted'):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Stone Free", "Residual"], digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\n")

evaluate_model("XGBOOST", y_test, y_pred_xgb)
evaluate_model("SVM", y_test, y_pred_svm)

print("XGBoost Best Params:")
print(xgb_grid.best_params_)
print("\nSVM Best Params:")
print(svm_grid.best_params_)

--- XGBOOST ---
Accuracy: 0.6481
Precision: 0.6596
Recall: 0.6481
F1 Score: 0.6537

Classification Report:
              precision    recall  f1-score   support

  Stone Free     0.7857    0.7674    0.7765        43
    Residual     0.1667    0.1818    0.1739        11

    accuracy                         0.6481        54
   macro avg     0.4762    0.4746    0.4752        54
weighted avg     0.6596    0.6481    0.6537        54

Confusion Matrix:
[[33 10]
 [ 9  2]]


--- SVM ---
Accuracy: 0.7963
Precision: 0.7896
Recall: 0.7963
F1 Score: 0.7926

Classification Report:
              precision    recall  f1-score   support

  Stone Free     0.8636    0.8837    0.8736        43
    Residual     0.5000    0.4545    0.4762        11

    accuracy                         0.7963        54
   macro avg     0.6818    0.6691    0.6749        54
weighted avg     0.7896    0.7963    0.7926        54

Confusion Matrix:
[[38  5]
 [ 6  5]]


XGBoost Best Params:
{'colsample_bytree': 0.9, 'gamma': 0,